# VERA-Base v. VERA-E

# 1. Imports & Configuration

In [1]:
import os
import random
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd

import pefile

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models

# Reproducibility
RANDOM_SEED = 101
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# 2. Paths & File Collection

In [2]:
# RW & Benign paths
ransomware_dir = Path("/mnt/mydrive/Datasets/Nithish Dataset/other")
benign_dir     = Path("/mnt/mydrive/Datasets/malimg/malware")

# For CNN images
benign_img_dir = Path("/mnt/mydrive/Datasets/Nithish Dataset/mw-img")
ransom_img_dir = Path("/mnt/mydrive/Datasets/Nithish Dataset/rw-img")

benign_img_dir.mkdir(parents=True, exist_ok=True)
ransom_img_dir.mkdir(parents=True, exist_ok=True)

def list_exe_files(root: Path) -> List[Path]:
    return [p for p in root.rglob("*") if p.is_file()]

rw_files_all = list_exe_files(ransomware_dir)
benign_files_all = list_exe_files(benign_dir)

len(rw_files_all), len(benign_files_all)

(40013, 57293)

## 2.1 Balance to 14,357 of each

In [3]:
TARGET_PER_CLASS = 40013

if len(benign_files_all) < TARGET_PER_CLASS:
    raise ValueError(f"Not enough benign files ({len(benign_files_all)}) to reach {TARGET_PER_CLASS}.")

if len(rw_files_all) < TARGET_PER_CLASS:
    raise ValueError(f"Not enough ransomware files ({len(rw_files_all)}) to reach {TARGET_PER_CLASS}.")

rw_files = random.sample(rw_files_all, TARGET_PER_CLASS)
benign_files = random.sample(benign_files_all, TARGET_PER_CLASS)

assert len(rw_files) == len(benign_files) == TARGET_PER_CLASS

print(f"{len(rw_files)} ransomware and {len(benign_files)} benign samples.")

40013 ransomware and 40013 benign samples.


# 3. Static Feature Extraction

In [4]:
import pefile
import lief
import numpy as np
from pathlib import Path

def section_entropy(data):
    if not data:
        return 0.0
    arr = np.frombuffer(data, dtype=np.uint8)
    probs = np.bincount(arr, minlength=256) / len(arr)
    probs = probs[probs > 0]
    return float(-(probs * np.log2(probs)).sum())

def extract_features_for_file(path: Path):
    feats = {}
    file_bytes = path.read_bytes()

    # ============================================================
    # Global byte-based features
    # ============================================================
    feats["file_size"] = len(file_bytes)
    feats["entropy"] = section_entropy(file_bytes)

    # 256-bin full histogram
    hist, _ = np.histogram(np.frombuffer(file_bytes, dtype=np.uint8),
                           bins=256, range=(0,256), density=True)
    for i, v in enumerate(hist):
        feats[f"byte_hist_{i}"] = float(v)

    # ============================================================
    # Load PE using pefile
    # ============================================================
    try:
        pe = pefile.PE(str(path), fast_load=False)
    except:
        # Fill with zeros for non-PE files
        return {k: 0 for k in feats}

    # ============================================================
    # PE Header Features
    # ============================================================
    header_fields = [
        "Machine", "NumberOfSections", "TimeDateStamp",
        "PointerToSymbolTable", "NumberOfSymbols", "SizeOfOptionalHeader",
        "Characteristics"
    ]

    for f in header_fields:
        feats[f"FILE_HEADER_{f}"] = getattr(pe.FILE_HEADER, f, 0)

    optional_fields = [
        "AddressOfEntryPoint", "ImageBase", "SectionAlignment",
        "FileAlignment", "MajorOperatingSystemVersion",
        "MinorOperatingSystemVersion", "MajorImageVersion",
        "MinorImageVersion", "SizeOfImage", "SizeOfHeaders",
        "DllCharacteristics", "LoaderFlags"
    ]

    for f in optional_fields:
        feats[f"OPTIONAL_HEADER_{f}"] = getattr(pe.OPTIONAL_HEADER, f, 0)

    # ============================================================
    # Section-level Features
    # ============================================================
    feats["num_sections"] = len(pe.sections)

    for i, sec in enumerate(pe.sections):
        name = sec.Name.rstrip(b'\x00').decode(errors="ignore")
        feats[f"section_{i}_name"] = name
        feats[f"section_{i}_entropy"] = section_entropy(sec.get_data())
        feats[f"section_{i}_raw_size"] = sec.SizeOfRawData
        feats[f"section_{i}_virtual_size"] = sec.Misc_VirtualSize

    # ============================================================
    # Imports / Exports
    # ============================================================
    try:
        imports = pe.DIRECTORY_ENTRY_IMPORT
        feats["num_import_dlls"] = len(imports)
        feats["num_imports_total"] = sum(len(dll.imports) for dll in imports)
    except:
        feats["num_import_dlls"] = 0
        feats["num_imports_total"] = 0

    try:
        exports = pe.DIRECTORY_ENTRY_EXPORT.symbols
        feats["num_exports"] = len(exports)
    except:
        feats["num_exports"] = 0

    # ============================================================
    # Resources (if any)
    # ============================================================
    try:
        resources = pe.DIRECTORY_ENTRY_RESOURCE.entries
        feats["num_resources"] = len(resources)
    except:
        feats["num_resources"] = 0

    # ============================================================
    # Overlay (extra data appended to PE)
    # ============================================================
    try:
        overlay_offset = pe.get_overlay_data_start_offset()
        if overlay_offset:
            feats["overlay_size"] = len(file_bytes) - overlay_offset
        else:
            feats["overlay_size"] = 0
    except:
        feats["overlay_size"] = 0

    # ============================================================
    # Rich Header (compiler metadata)
    # ============================================================
    try:
        rh = pe.parse_rich_header()
        feats["rich_header_hash"] = rh.get("hash", 0)
    except:
        feats["rich_header_hash"] = 0

    return feats

In [5]:
import pandas as pd
from pathlib import Path

# Load your existing full dataset
df_existing = pd.read_csv("/mnt/mydrive/Datasets/Nithish Dataset/feature_dataset.csv", low_memory=False)

# Reconstruct the original file lists
rw_files = [Path(p) for p in df_existing[df_existing["label"] == 1]["path"].tolist()]
benign_files = [Path(p) for p in df_existing[df_existing["label"] == 0]["path"].tolist()]

print("Recovered ransomware files:", len(rw_files))
print("Recovered benign files:", len(benign_files))


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/mydrive/Datasets/Nithish Dataset/feature_dataset.csv'

## 3.1 Extract features

In [6]:
def build_feature_dataframe(files: List[Path], label: int) -> pd.DataFrame:
    rows = []
    for i, p in enumerate(files):
        if i % 1000 == 0:
            print(f"Processed {i}/{len(files)} files...")
        feats = extract_features_for_file(p)
        feats["label"] = label
        feats["path"] = str(p)
        rows.append(feats)
    return pd.DataFrame(rows)

df_rw = build_feature_dataframe(rw_files, label=1)
df_benign = build_feature_dataframe(benign_files, label=0)

df = pd.concat([df_rw, df_benign], ignore_index=True)

# Fill missing PE features with 0 and ensure consistent columns
df.fillna(0, inplace=True)

# ---- DROP COLUMNS MISSING IN PACKED DATASET ----
cols_to_drop = [
    "section_5_entropy","section_5_name","section_5_raw_size","section_5_virtual_size",
    "section_6_entropy","section_6_name","section_6_raw_size","section_6_virtual_size",
    "section_7_entropy","section_7_name","section_7_raw_size","section_7_virtual_size",
    "section_8_entropy","section_8_name","section_8_raw_size","section_8_virtual_size",
    "section_9_entropy","section_9_name","section_9_raw_size","section_9_virtual_size",
    "section_10_entropy","section_10_name","section_10_raw_size","section_10_virtual_size",
    "section_11_entropy","section_11_name","section_11_raw_size","section_11_virtual_size",
    "section_12_entropy","section_12_name","section_12_raw_size","section_12_virtual_size",
    "section_13_entropy","section_13_name","section_13_raw_size","section_13_virtual_size",
    "section_14_entropy","section_14_name","section_14_raw_size","section_14_virtual_size",
    "section_15_entropy","section_15_name","section_15_raw_size","section_15_virtual_size",
    "section_16_entropy","section_16_name","section_16_raw_size","section_16_virtual_size",
    "section_17_entropy","section_17_name","section_17_raw_size","section_17_virtual_size",
    "section_18_entropy","section_18_name","section_18_raw_size","section_18_virtual_size",
    "section_19_entropy","section_19_name","section_19_raw_size","section_19_virtual_size",
    "section_20_entropy","section_20_name","section_20_raw_size","section_20_virtual_size",
    "section_21_entropy","section_21_name","section_21_raw_size","section_21_virtual_size",
    "section_22_entropy","section_22_name","section_22_raw_size","section_22_virtual_size",
    "section_23_entropy","section_23_name","section_23_raw_size","section_23_virtual_size",
    "section_24_entropy","section_24_name","section_24_raw_size","section_24_virtual_size",
    "section_25_entropy","section_25_name","section_25_raw_size","section_25_virtual_size",
    "section_26_entropy","section_26_name","section_26_raw_size","section_26_virtual_size",
    "section_27_entropy","section_27_name","section_27_raw_size","section_27_virtual_size",
    "section_28_entropy","section_28_name","section_28_raw_size","section_28_virtual_size",
    "section_29_entropy","section_29_name","section_29_raw_size","section_29_virtual_size",
    "section_30_entropy","section_30_name","section_30_raw_size","section_30_virtual_size",
    "section_31_entropy","section_31_name","section_31_raw_size","section_31_virtual_size",
    "section_32_entropy","section_32_name","section_32_raw_size","section_32_virtual_size",
    "section_33_entropy","section_33_name","section_33_raw_size","section_33_virtual_size",
]

df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)


print(df.shape)
df.head()
df.to_csv("/mnt/mydrive/Datasets/Nithish Dataset/feature_dataset.csv", index=False)

Processed 0/40013 files...
Processed 1000/40013 files...
Processed 2000/40013 files...
Processed 3000/40013 files...
Processed 4000/40013 files...
Processed 5000/40013 files...
Processed 6000/40013 files...
Processed 7000/40013 files...
Processed 8000/40013 files...
Processed 9000/40013 files...
Processed 10000/40013 files...
Processed 11000/40013 files...
Processed 12000/40013 files...
Processed 13000/40013 files...
Processed 14000/40013 files...
Processed 15000/40013 files...
Processed 16000/40013 files...
Processed 17000/40013 files...
Processed 18000/40013 files...
Processed 19000/40013 files...
Processed 20000/40013 files...
Processed 21000/40013 files...
Processed 22000/40013 files...
Processed 23000/40013 files...
Processed 24000/40013 files...
Processed 25000/40013 files...
Processed 26000/40013 files...
Processed 27000/40013 files...
Processed 28000/40013 files...
Processed 29000/40013 files...
Processed 30000/40013 files...
Processed 31000/40013 files...
Processed 32000/40013

# 4. Train/Test Split & Scaling

In [7]:
df = pd.read_csv("/mnt/mydrive/Datasets/Nithish Dataset/feature_dataset.csv")

from sklearn.preprocessing import OrdinalEncoder

# Identify PE section name columns
name_cols = [c for c in df.columns if c.startswith("section_") and c.endswith("_name")]

# 1. Force all section names to strings to avoid mixed-type errors
df[name_cols] = df[name_cols].astype(str)

# 2. Ordinal encode section names
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df[name_cols] = enc.fit_transform(df[name_cols])

# 3. Convert all other columns to numeric
for col in df.columns:
    if col not in name_cols and col not in ["label", "path"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df.fillna(0, inplace=True)

# 4. Build ML matrices
feature_cols = [c for c in df.columns if c not in ["label", "path"]]
X = df[feature_cols].values.astype(float)
y = df["label"].values

# 5. Split + scale
X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    X, y, df["path"].values,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df["label"]
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape


/tmp/ipykernel_3335636/1775810447.py:1: DtypeWarning: Columns (306,310,314,318,322,326,330,334,338,342,346,350,354,358,362,366,370,374,378,382,386,390,394,398,402,406,410,414,418,422,426,430,434,438,442,446,450,454,458,462,466,470,474,478,482,486,490,494,498,502,506,510,514,518,522,526,530,534,538,542,546,550,554,558,562,566,570,574,578,582,586,590,594,598,602,606,610,614,618,622,626,630,634,638,642,646,650,654,658,662,666,670,674) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/mnt/mydrive/Datasets/Nithish Dataset/feature_dataset.csv")


((64020, 676), (16006, 676))

# 5. Classical ML Models

In [8]:
def evaluate_binary_classifier(model, X_tr, y_tr, X_te, y_te, name: str) -> Dict[str, float]:
    model.fit(X_tr, y_tr)

    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)

    if hasattr(model, "predict_proba"):
        y_te_proba = model.predict_proba(X_te)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_te)
        y_te_proba = 1 / (1 + np.exp(-scores))
    else:
        y_te_proba = y_te_pred.astype(float)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_te, y_te_pred),
        "precision": precision_score(y_te, y_te_pred, zero_division=0),
        "recall": recall_score(y_te, y_te_pred, zero_division=0),
        "f1": f1_score(y_te, y_te_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_te, y_te_proba),
    }

    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        if k != "model":
            print(f"{k}: {v:.4f}")

    return metrics, y_te_pred, y_te_proba

results = {}
predictions_test = {}   # store per-model test predictions
probabilities_test = {} # store per-model test probabilities

## 5.1 Random Forest

In [9]:
rf = RandomForestClassifier(
    n_estimators=800,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1
)


rf_metrics, rf_y_pred, rf_y_proba = evaluate_binary_classifier(
    rf, X_train_scaled, y_train, X_test_scaled, y_test, "RandomForest"
)
results["RandomForest"] = rf_metrics
predictions_test["RandomForest"] = rf_y_pred
probabilities_test["RandomForest"] = rf_y_proba


=== RandomForest ===
accuracy: 0.9998
precision: 0.9998
recall: 0.9998
f1: 0.9998
roc_auc: 1.0000


## 5.2 Support Vector Machine (Radial Basis Function)

In [10]:
svm = SVC(
    kernel="rbf",
    C=10,
    gamma=0.01,
    probability=True,
    class_weight="balanced",
    random_state=RANDOM_SEED
)


svm_metrics, svm_y_pred, svm_y_proba = evaluate_binary_classifier(
    svm, X_train_scaled, y_train, X_test_scaled, y_test, "SVM_RBF"
)
results["SVM_RBF"] = svm_metrics
predictions_test["SVM_RBF"] = svm_y_pred
probabilities_test["SVM_RBF"] = svm_y_proba


=== SVM_RBF ===
accuracy: 0.9803
precision: 0.9901
recall: 0.9703
f1: 0.9801
roc_auc: 0.9990


## 5.3 Multi Layer Perceptron

In [11]:
mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation="relu",
    solver="adam",
    alpha=1e-5,
    learning_rate="adaptive",
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=RANDOM_SEED
)


mlp_metrics, mlp_y_pred, mlp_y_proba = evaluate_binary_classifier(
    mlp, X_train_scaled, y_train, X_test_scaled, y_test, "MLP"
)
results["MLP"] = mlp_metrics
predictions_test["MLP"] = mlp_y_pred
probabilities_test["MLP"] = mlp_y_proba


=== MLP ===
accuracy: 0.9975
precision: 0.9975
recall: 0.9975
f1: 0.9975
roc_auc: 0.9997


## 5.4 XGBoost & Final Model Results

In [12]:
xgb = XGBClassifier(
    n_estimators=600,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=1.0,
    reg_lambda=2.0,
    gamma=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=1.0,
    random_state=RANDOM_SEED
)


xgb_metrics, xgb_y_pred, xgb_y_proba = evaluate_binary_classifier(
    xgb, X_train_scaled, y_train, X_test_scaled, y_test, "XGBoost"
)
results["XGBoost"] = xgb_metrics
predictions_test["XGBoost"] = xgb_y_pred
probabilities_test["XGBoost"] = xgb_y_proba

results_df = pd.DataFrame(results).T
results_df


=== XGBoost ===
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1: 1.0000
roc_auc: 1.0000


,model,accuracy,precision,recall,f1,roc_auc
RandomForest,RandomForest,0.99975,0.99975,0.99975,0.99975,1.0
SVM_RBF,SVM_RBF,0.980257,0.990055,0.970261,0.980058,0.998957
MLP,MLP,0.997501,0.997501,0.997501,0.997501,0.999746
XGBoost,XGBoost,1.0,1.0,1.0,1.0,1.0


# 6. EXE → PNG Conversion (for CNN)

In [20]:
IMG_SIZE = 256  # 256x256 images

def exe_to_image_bytes(path: Path, img_size: int = IMG_SIZE) -> np.ndarray:
    with open(path, "rb") as f:
        data = f.read()

    arr = np.frombuffer(data, dtype=np.uint8)

    target_len = img_size * img_size
    if len(arr) < target_len:
        # pad with zeros
        pad_len = target_len - len(arr)
        arr = np.concatenate([arr, np.zeros(pad_len, dtype=np.uint8)])
    else:
        # truncate
        arr = arr[:target_len]

    img = arr.reshape((img_size, img_size))
    return img

def save_exe_as_png(src_paths: List[Path], dst_dir: Path, label_str: str) -> pd.DataFrame:
    dst_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, src in enumerate(src_paths):
        img_arr = exe_to_image_bytes(src, IMG_SIZE)
        img = Image.fromarray(img_arr, mode="L")  # grayscale

        # Unique filename
        out_name = f"{label_str}_{i:06d}.png"
        out_path = dst_dir / out_name
        img.save(out_path)

        rows.append({
            "original_path": str(src),
            "image_path": str(out_path),
            "label": 1 if label_str == "rw" else 0
        })
    return pd.DataFrame(rows)

df_img_rw = save_exe_as_png(rw_files, ransom_img_dir, "rw")
df_img_benign = save_exe_as_png(benign_files, benign_img_dir, "benign")

df_img = pd.concat([df_img_rw, df_img_benign], ignore_index=True)
df_img.head(), df_img["label"].value_counts()

(                                       original_path  \
 0  /mnt/mydrive/Datasets/Nithish Dataset/other/Tr...   
 1  /mnt/mydrive/Datasets/Nithish Dataset/other/Tr...   
 2  /mnt/mydrive/Datasets/Nithish Dataset/other/Tr...   
 3  /mnt/mydrive/Datasets/Nithish Dataset/other/Tr...   
 4  /mnt/mydrive/Datasets/Nithish Dataset/other/Tr...   
 
                                           image_path  label  
 0  /mnt/mydrive/Datasets/Nithish Dataset/rw-img/r...      1  
 1  /mnt/mydrive/Datasets/Nithish Dataset/rw-img/r...      1  
 2  /mnt/mydrive/Datasets/Nithish Dataset/rw-img/r...      1  
 3  /mnt/mydrive/Datasets/Nithish Dataset/rw-img/r...      1  
 4  /mnt/mydrive/Datasets/Nithish Dataset/rw-img/r...      1  ,
 label
 1    14357
 0    14357
 Name: count, dtype: int64)

# 7. CNN Model for Image Classification

In [16]:
class RansomwareImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]
        label = int(row["label"])

        img = Image.open(img_path).convert("L")  # grayscale
        if self.transform is not None:
            img = self.transform(img)

        return img, label

# Train/test split for images (keep consistent with prior split via original paths)
test_paths_set = set(paths_test)

df_img["is_test"] = df_img["original_path"].isin(test_paths_set)
df_img_train = df_img[~df_img["is_test"]].copy()
df_img_test  = df_img[df_img["is_test"]].copy()

print(df_img_train["label"].value_counts())
print(df_img_test["label"].value_counts())

NameError: name 'df_img' is not defined

## 7.1 Transforms & Dataloaders

In [22]:
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_dataset = RansomwareImageDataset(df_img_train, transform=transform_train)
test_dataset  = RansomwareImageDataset(df_img_test, transform=transform_test)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

## 7.2 CNN Architecture

In [23]:
class RansomwareCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


cnn_model = RansomwareCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3)

## 7.3 Train CNN

In [24]:
def train_cnn(model, train_loader, test_loader, criterion, optimizer, epochs=15):
    model.to(device)
    history = {"train_loss": [], "test_loss": [], "test_acc": []}

    best_acc = 0.0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.float().to(device)

            optimizer.zero_grad()
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        # Evaluate
        model.eval()
        test_loss = 0.0
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.float().to(device)
                outputs = model(images).squeeze(1)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * images.size(0)

                probs = torch.sigmoid(outputs).cpu().numpy()
                all_probs.append(probs)
                all_labels.append(labels.cpu().numpy())

        test_loss /= len(test_loader.dataset)
        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)
        preds = (all_probs >= 0.5).astype(int)
        acc = accuracy_score(all_labels, preds)

        print(f"Epoch {epoch}/{epochs} - Train Loss: {train_loss:.4f} "
              f"- Test Loss: {test_loss:.4f} - Test Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            best_state = model.state_dict()

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best model with accuracy: {best_acc:.4f}")

    return history

### Run CNN

In [25]:
cnn_history = train_cnn(cnn_model, train_loader, test_loader, criterion, optimizer, epochs=10)

Epoch 1/10 - Train Loss: 0.9347 - Test Loss: 0.1236 - Test Acc: 0.9586
Epoch 2/10 - Train Loss: 0.1377 - Test Loss: 0.0875 - Test Acc: 0.9754
Epoch 3/10 - Train Loss: 0.1106 - Test Loss: 0.0850 - Test Acc: 0.9746
Epoch 4/10 - Train Loss: 0.0965 - Test Loss: 0.0897 - Test Acc: 0.9753
Epoch 5/10 - Train Loss: 0.0918 - Test Loss: 0.1008 - Test Acc: 0.9664
Epoch 6/10 - Train Loss: 0.0826 - Test Loss: 0.0766 - Test Acc: 0.9774
Epoch 7/10 - Train Loss: 0.0842 - Test Loss: 0.1137 - Test Acc: 0.9735
Epoch 8/10 - Train Loss: 0.0809 - Test Loss: 0.0799 - Test Acc: 0.9786
Epoch 9/10 - Train Loss: 0.0819 - Test Loss: 0.1008 - Test Acc: 0.9756
Epoch 10/10 - Train Loss: 0.0748 - Test Loss: 0.0907 - Test Acc: 0.9781

Restored best model with accuracy: 0.9786


## 7.4 Eval & Metrics

In [15]:
def evaluate_cnn(model, data_loader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images).squeeze(1)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.cpu().numpy())
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    preds = (all_probs >= 0.5).astype(int)
    return all_labels, all_probs

y_test_cnn, y_proba_cnn = evaluate_cnn(cnn_model, test_loader)

cnn_metrics = {
    "model": "CNN",
    "accuracy": accuracy_score(y_test_cnn, (y_proba_cnn >= 0.5).astype(int)),
    "precision": precision_score(y_test_cnn, (y_proba_cnn >= 0.5).astype(int), zero_division=0),
    "recall": recall_score(y_test_cnn, (y_proba_cnn >= 0.5).astype(int), zero_division=0),
    "f1": f1_score(y_test_cnn, (y_proba_cnn >= 0.5).astype(int), zero_division=0),
    "roc_auc": roc_auc_score(y_test_cnn, y_proba_cnn)
}
cnn_metrics

results["CNN"] = cnn_metrics
results_df = pd.DataFrame(results).T
results_df

NameError: name 'cnn_model' is not defined

# 8. Evasion Techniques

## 8.1 Packing

In [27]:
import subprocess
import shutil
from pathlib import Path
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import pefile
import pandas as pd

df = pd.read_csv("/mnt/mydrive/Datasets/Nithish Dataset/feature_dataset.csv")

all_paths = [Path(p) for p in df["path"].tolist()]
rw_files = [p for p, lbl in zip(all_paths, df["label"]) if lbl == 1]
benign_files = [p for p, lbl in zip(all_paths, df["label"]) if lbl == 0]

print("Loaded:", len(rw_files), "ransomware,", len(benign_files), "benign")

# Output directory for UPX dataset
OUTPUT_DIR = Path("/mnt/mydrive/Datasets/Nithish Dataset/upx_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def is_valid_pe(file_path: Path) -> bool:
    """Returns True if file is a valid PE file."""
    try:
        pe = pefile.PE(str(file_path))
        pe.close()
        return True
    except Exception:
        return False


def is_upx_packed(path: Path) -> bool:
    """Detect if the file is already UPX-packed (by section names)."""
    try:
        pe = pefile.PE(str(path))
        for section in pe.sections:
            name = section.Name.rstrip(b"\x00").decode("utf-8", "ignore")
            if name in ("UPX0", "UPX1", "UPX2"):
                return True
        return False
    except Exception:
        return False


def pack_with_upx(file_path: Path) -> str:
    """
    Keep only:
      - files that were already UPX-packed (copied),
      - files that successfully pack with UPX and remain valid PE.

    Skip everything else.
    """
    out_filename = f"upx_{file_path.name}"
    out_path = OUTPUT_DIR / out_filename

    try:
        # 1) Already UPX-packed, copy it, done.
        if is_upx_packed(file_path):
            shutil.copy2(file_path, out_path)
            return f"COPIED_ALREADY_UPX: {file_path.name}"

        # 2) If it's not a valid PE -> skip entirely.
        if not is_valid_pe(file_path):
            return f"SKIP_NON_PE: {file_path.name}"

        # 3) It's a valid PE and not UPX-packed: try to pack it.
        shutil.copy2(file_path, out_path)

        result = subprocess.run(
            ["upx", "--best", "--lzma", str(out_path)],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        # If UPX failed, remove file, skip.
        if result.returncode != 0:
            out_path.unlink(missing_ok=True)
            return f"SKIP_UPX_FAIL: {file_path.name}"

        # Validate PE after packing
        if not is_valid_pe(out_path):
            out_path.unlink(missing_ok=True)
            return f"SKIP_INVALID_AFTER_PACK: {file_path.name}"

        # Success: packed and valid.
        return f"OK_PACKED: {file_path.name}"

    except Exception as e:
        out_path.unlink(missing_ok=True)
        return f"ERROR: {file_path.name}: {e}"


# Only pack ransomware files
def pack_all_with_progress(rw_files):
    print(f"Packing {len(rw_files)} ransomware files with UPX...")

    workers = max(cpu_count() - 1, 1)

    with Pool(workers) as pool:
        for _ in tqdm(
            pool.imap_unordered(pack_with_upx, rw_files),
            total=len(rw_files),
            desc="UPX packing (RW only)",
            ncols=85
        ):
            pass

pack_all_with_progress(rw_files)


Loaded: 14357 ransomware, 14357 benign
Packing 14357 ransomware files with UPX...


/tmp/ipykernel_282008/3864100005.py:9: DtypeWarning: Columns (302,306,310,314,318,322,326,330,334,338,342,346,350,354,358,362,366,370,374,378,382,386,390,394,398,402,406,410,414,418) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/mnt/mydrive/Datasets/Nithish Dataset/feature_dataset.csv")
UPX packing (RW only): 100%|███████████████████| 14357/14357 [12:34<00:00, 19.04it/s]


## Base vs. Packed Performance

In [20]:
import pandas as pd

def make_row(model, version, metrics):
    row = {"model": model, "version": version}
    row.update(metrics)
    return row


rows = [
    make_row("RandomForest", "unpacked", results["RandomForest"]),
    make_row("RandomForest", "packed",   rf_eval_metrics),

    make_row("SVM_RBF", "unpacked", results["SVM_RBF"]),
    make_row("SVM_RBF", "packed",   svm_eval_metrics),

    make_row("MLP", "unpacked", results["MLP"]),
    make_row("MLP", "packed",   mlp_eval_metrics),

    make_row("XGBoost", "unpacked", results["XGBoost"]),
    make_row("XGBoost", "packed",   xgb_eval_metrics),
]

df_all = pd.DataFrame(rows)
df_all = df_all.sort_values(["model", "version"]).reset_index(drop=True)

# Table A
perf_cols = ["accuracy", "precision", "recall", "f1"]

table_A = df_all[["model", "version"] + perf_cols]

print("Core Performance Metrics")
display(table_A)

# Table B
attack_cols = ["entropy_mean", "ASR"]

table_B = df_all[df_all["version"] == "packed"][["model", "version"] + attack_cols]

print("Packed-Only Attack Metrics")
display(table_B)

# Table C
delta_rows = []

for model in ["RandomForest", "SVM_RBF", "MLP", "XGBoost"]:
    row_un = df_all[(df_all.model == model) & (df_all.version == "unpacked")].iloc[0]
    row_pk = df_all[(df_all.model == model) & (df_all.version == "packed")].iloc[0]

    delta = {"model": model, "version": "delta"}

    for col in perf_cols:  # Only core metrics
        if pd.isna(row_pk[col]) or pd.isna(row_un[col]):
            delta[col] = None
        else:
            delta[col] = row_pk[col] - row_un[col]

    delta_rows.append(delta)

table_C = pd.DataFrame(delta_rows)

print("=== Performance Change ===")
display(table_C)


Core Performance Metrics


,model,version,accuracy,precision,recall,f1
0,MLP,packed,0.932265,0.996400,0.867642,0.927574
1,MLP,unpacked,0.974752,0.982655,0.966574,0.974548
2,RandomForest,packed,0.999478,1.000000,0.998955,0.999477
3,RandomForest,unpacked,0.986070,0.995739,0.976323,0.985935
4,SVM_RBF,packed,0.878461,0.993191,0.762104,0.862436
5,SVM_RBF,unpacked,0.974055,0.984347,0.963440,0.973781
6,XGBoost,packed,0.999826,0.999652,1.000000,0.999826
7,XGBoost,unpacked,0.985896,0.994332,0.977368,0.985777


Packed-Only Attack Metrics


,model,version,entropy_mean,ASR
0,MLP,packed,0.242143,0.063741
2,RandomForest,packed,0.178398,0.001045
4,SVM_RBF,packed,0.279447,0.117729
6,XGBoost,packed,0.059770,0.000000


=== Performance Change ===


,model,version,accuracy,precision,recall,f1
0,RandomForest,delta,0.013408,0.004261,0.022632,0.013542
1,SVM_RBF,delta,-0.095595,0.008844,-0.201336,-0.111346
2,MLP,delta,-0.042487,0.013745,-0.098932,-0.046974
3,XGBoost,delta,0.013930,0.005320,0.022632,0.014049


# %=====================

## 8.2 Encrypt Ransomware

## 8.4 Image Steganography

# Test Packing

In [1]:
import pandas as pd
import numpy as np
import subprocess
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pefile

# ============================================================
# 0. CONFIG
# ============================================================
RANDOM_SEED = 101  # must match your base training code

BASE_CSV = "/mnt/mydrive/Datasets/Nithish Dataset/feature_dataset_fixed.csv"
UPX_OUT_DIR = Path("/mnt/mydrive/Datasets/Nithish Dataset/upx_test_ransom")
UPX_FEATURE_CSV = "/mnt/mydrive/Datasets/Nithish Dataset/upx_feature_dataset_raw.csv"
MISSING_FEATURES_CSV = "/mnt/mydrive/Datasets/Nithish Dataset/features_to_remove_from_base.csv"

UPX_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. LOAD BASE DATASET + SPLIT TRAIN/TEST
# ============================================================
df_base = pd.read_csv(BASE_CSV, low_memory=False)
print("Loaded base dataset:", df_base.shape)

all_paths = df_base["path"].values
y = df_base["label"].values

# feature columns actually used for ML in the base pipeline
feature_cols = [c for c in df_base.columns if c not in ["label", "path"]]
X = df_base[feature_cols].values

X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    X,
    y,
    all_paths,
    test_size=0.2,
    random_state=RANDOM_SEED,  # IMPORTANT: same seed as base training
    stratify=y
)

print("Train/Test split done.")
print("Test set size:", len(y_test))

# ============================================================
# 2. IDENTIFY TEST RANSOMWARE + PACK WITH UPX
# ============================================================
rw_mask = (y_test == 1)
rw_test_paths = np.array(paths_test)[rw_mask]
print("Ransomware in test set:", len(rw_test_paths))


def pack_upx_single(path: Path) -> Path | None:
    """
    Pack a single file with UPX into UPX_OUT_DIR as 'upx_<name>'.
    If the packed file already exists, reuse it.
    """
    out_path = UPX_OUT_DIR / f"upx_{path.name}"

    # reuse if already exists
    if out_path.exists():
        return out_path

    try:
        shutil.copy2(path, out_path)
        result = subprocess.run(
            ["upx", "--best", "--lzma", str(out_path)],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=120,
        )
        if result.returncode != 0:
            # packing failed; clean up and skip
            out_path.unlink(missing_ok=True)
            return None
        return out_path
    except Exception as e:
        print(f"[ERROR] packing {path}: {e}")
        out_path.unlink(missing_ok=True)
        return None


print("Packing ransomware (or reusing existing packed files)...")
packed_paths: list[Path] = []

for p_str in tqdm(rw_test_paths, desc="UPX Packing", ncols=90):
    p = Path(p_str)
    packed = pack_upx_single(p)
    if packed:
        packed_paths.append(packed)

print("\nPacked or reused:", len(packed_paths))
if len(packed_paths) == 0:
    raise RuntimeError("No test ransomware could be packed. Check UPX / paths.")

# ============================================================
# 3. FEATURE EXTRACTION (SAME AS BASE EXTRACTOR)
# ============================================================
def section_entropy(data: bytes) -> float:
    if not data:
        return 0.0
    arr = np.frombuffer(data, dtype=np.uint8)
    probs = np.bincount(arr, minlength=256) / len(arr)
    probs = probs[probs > 0]
    return float(-(probs * np.log2(probs)).sum())


def extract_features_for_file(path: Path) -> dict:
    feats: dict[str, float | int | str] = {}
    file_bytes = path.read_bytes()

    # -------- global byte features --------
    feats["file_size"] = len(file_bytes)
    feats["entropy"] = section_entropy(file_bytes)

    hist, _ = np.histogram(
        np.frombuffer(file_bytes, dtype=np.uint8),
        bins=256,
        range=(0, 256),
        density=True,
    )
    for i, v in enumerate(hist):
        feats[f"byte_hist_{i}"] = float(v)

    # -------- PE features --------
    try:
        pe = pefile.PE(str(path), fast_load=False)
    except Exception:
        # if not a valid PE, just return the byte-level features we already have
        return feats

    # FILE_HEADER
    header_fields = [
        "Machine",
        "NumberOfSections",
        "TimeDateStamp",
        "PointerToSymbolTable",
        "NumberOfSymbols",
        "SizeOfOptionalHeader",
        "Characteristics",
    ]
    for f in header_fields:
        feats[f"FILE_HEADER_{f}"] = getattr(pe.FILE_HEADER, f, 0)

    # OPTIONAL_HEADER
    optional_fields = [
        "AddressOfEntryPoint",
        "ImageBase",
        "SectionAlignment",
        "FileAlignment",
        "MajorOperatingSystemVersion",
        "MinorOperatingSystemVersion",
        "MajorImageVersion",
        "MinorImageVersion",
        "SizeOfImage",
        "SizeOfHeaders",
        "DllCharacteristics",
        "LoaderFlags",
    ]
    for f in optional_fields:
        feats[f"OPTIONAL_HEADER_{f}"] = getattr(pe.OPTIONAL_HEADER, f, 0)

    # Sections
    feats["num_sections"] = len(pe.sections)
    for i, sec in enumerate(pe.sections):
        name = sec.Name.rstrip(b"\x00").decode(errors="ignore")
        feats[f"section_{i}_name"] = name
        feats[f"section_{i}_entropy"] = section_entropy(sec.get_data())
        feats[f"section_{i}_raw_size"] = sec.SizeOfRawData
        feats[f"section_{i}_virtual_size"] = sec.Misc_VirtualSize

    # Imports
    try:
        imports = pe.DIRECTORY_ENTRY_IMPORT
        feats["num_import_dlls"] = len(imports)
        feats["num_imports_total"] = sum(len(dll.imports) for dll in imports)
    except Exception:
        feats["num_import_dlls"] = 0
        feats["num_imports_total"] = 0

    # Exports
    try:
        exports = pe.DIRECTORY_ENTRY_EXPORT.symbols
        feats["num_exports"] = len(exports)
    except Exception:
        feats["num_exports"] = 0

    # Resources
    try:
        resources = pe.DIRECTORY_ENTRY_RESOURCE.entries
        feats["num_resources"] = len(resources)
    except Exception:
        feats["num_resources"] = 0

    # Overlay
    try:
        overlay_offset = pe.get_overlay_data_start_offset()
        if overlay_offset:
            feats["overlay_size"] = len(file_bytes) - overlay_offset
        else:
            feats["overlay_size"] = 0
    except Exception:
        feats["overlay_size"] = 0

    # Rich header
    try:
        rh = pe.parse_rich_header()
        feats["rich_header_hash"] = rh.get("hash", 0)
    except Exception:
        feats["rich_header_hash"] = 0

    return feats


# ============================================================
# 4. EXTRACT FEATURES FOR *PACKED* TEST RANSOMWARE
# ============================================================
packed_rows: list[dict] = []

print("Extracting features from packed ransomware...")

for p in tqdm(packed_paths, desc="Feature extraction", ncols=90):
    feats = extract_features_for_file(p)
    feats["path"] = str(p)
    feats["original_name"] = p.name.replace("upx_", "", 1)
    feats["label"] = 1  # still ransomware
    packed_rows.append(feats)

df_packed_raw = pd.DataFrame(packed_rows)
df_packed_raw.fillna(0, inplace=True)

print("Raw packed feature shape:", df_packed_raw.shape)

# ============================================================
# 5. SAVE RAW PACKED FEATURE CSV
# ============================================================
df_packed_raw.to_csv(UPX_FEATURE_CSV, index=False)
print("Saved RAW packed features to:", UPX_FEATURE_CSV)

# ============================================================
# 6. FIND FEATURES THAT EXIST IN BASE BUT NOT IN PACKED
# ============================================================
packed_features = set(df_packed_raw.columns) - {"label", "path", "original_name"}
base_features = set(feature_cols)

missing_cols = sorted(base_features - packed_features)

print("\n===== FEATURES MISSING IN PACKED FILES (PRESENT IN BASE ONLY) =====")
for col in missing_cols:
    print(col)
print("Total missing:", len(missing_cols))

pd.DataFrame({"missing_features": missing_cols}).to_csv(
    MISSING_FEATURES_CSV, index=False
)

print("\nSaved missing feature list to:", MISSING_FEATURES_CSV)
print("\n=== SCRIPT COMPLETE ===")

Loaded base dataset: (28714, 422)
Train/Test split done.
Test set size: 5743
Ransomware in test set: 2872
Packing ransomware (or reusing existing packed files)...


UPX Packing: 100%|████████████████████████████████████| 2872/2872 [01:55<00:00, 24.88it/s]



Packed or reused: 1349
Extracting features from packed ransomware...


Feature extraction: 100%|█████████████████████████████| 1349/1349 [00:47<00:00, 28.13it/s]


Raw packed feature shape: (1349, 307)
Saved RAW packed features to: /mnt/mydrive/Datasets/Nithish Dataset/upx_feature_dataset_raw.csv

===== FEATURES MISSING IN PACKED FILES (PRESENT IN BASE ONLY) =====
section_10_entropy
section_10_name
section_10_raw_size
section_10_virtual_size
section_11_entropy
section_11_name
section_11_raw_size
section_11_virtual_size
section_12_entropy
section_12_name
section_12_raw_size
section_12_virtual_size
section_13_entropy
section_13_name
section_13_raw_size
section_13_virtual_size
section_14_entropy
section_14_name
section_14_raw_size
section_14_virtual_size
section_15_entropy
section_15_name
section_15_raw_size
section_15_virtual_size
section_16_entropy
section_16_name
section_16_raw_size
section_16_virtual_size
section_17_entropy
section_17_name
section_17_raw_size
section_17_virtual_size
section_18_entropy
section_18_name
section_18_raw_size
section_18_virtual_size
section_19_entropy
section_19_name
section_19_raw_size
section_19_virtual_size
section

# test test

In [14]:
import subprocess
import shutil
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load base schema (already column-matched!)
# ------------------------------------------------------------
df_base = pd.read_csv("/mnt/mydrive/Datasets/Nithish Dataset/feature_dataset.csv",
                      low_memory=False)

# same feature columns your models were trained on
feature_cols = [c for c in df_base.columns
                if c not in ["label", "path", "original_name"]]

# section name columns (need encoding)
name_cols = [c for c in feature_cols
             if c.startswith("section_") and c.endswith("_name")]

# enc must already exist from your training code:
# enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
# enc.fit( ... on base name_cols ... )

# ------------------------------------------------------------
# 1. Identify ransomware from TEST SET only
# ------------------------------------------------------------
rw_test_mask = (y_test == 1)
rw_test_paths = np.array(paths_test)[rw_test_mask]

print("Ransomware in test set:", len(rw_test_paths))

UPX_OUT = Path("/mnt/mydrive/Datasets/Nithish Dataset/upx_test_rw")
UPX_OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Pack files (skip if already packed)
# ------------------------------------------------------------
def pack_upx_single(path: Path):
    out_path = UPX_OUT / f"upx_{path.name}"
    if out_path.exists():
        return out_path

    try:
        shutil.copy2(path, out_path)
        result = subprocess.run(
            ["upx", "--best", "--lzma", str(out_path)],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, timeout=120
        )
        if result.returncode != 0:
            out_path.unlink(missing_ok=True)
            return None
        return out_path
    except Exception:
        out_path.unlink(missing_ok=True)
        return None

packed_paths = []
print("Packing ransomware...")

for p_str in rw_test_paths:
    packed = pack_upx_single(Path(p_str))
    if packed:
        packed_paths.append(packed)

print("Packed or reused:", len(packed_paths))


# ------------------------------------------------------------
# 3. Extract features for packed files
# ------------------------------------------------------------
packed_rows = []
for p in packed_paths:
    feats = extract_features_for_file(p)  # SAME extractor as base
    feats["label"] = 1
    feats["path"] = str(p)
    feats["original_name"] = p.name.replace("upx_", "", 1)
    packed_rows.append(feats)

df_packed = pd.DataFrame(packed_rows)
df_packed.fillna(0, inplace=True)

# ------------------------------------------------------------
# 3b. Force df_packed to have SAME columns as base
# ------------------------------------------------------------
df_packed = df_packed.set_index("original_name")

# add any missing feature columns (filled with 0), keep order
df_packed = df_packed.reindex(columns=feature_cols + ["path", "label"],
                              fill_value=0)

# ------------------------------------------------------------
# 3c. ENCODE section_*_name columns using SAME encoder as base
# ------------------------------------------------------------
if name_cols:  # just in case
    # cast to string then transform with enc from your training code
    df_packed[name_cols] = df_packed[name_cols].astype(str)
    df_packed[name_cols] = enc.transform(df_packed[name_cols])

# everything except path/label is now numeric and aligned with feature_cols

# ------------------------------------------------------------
# 4. Build packed test feature matrix (same columns as base)
# ------------------------------------------------------------
X_test_packed_scaled = X_test_scaled.copy()
paired_mask = np.zeros(len(y_test), dtype=bool)

for i, (is_rw, path) in enumerate(zip(y_test == 1, paths_test)):
    if not is_rw:
        continue

    base = Path(path).name

    if base in df_packed.index:
        # ---- USE PACKED FEATURES ----
        valid_cols = [c for c in feature_cols if c in df_packed.columns]

        packed_feats = (
            df_packed.loc[base, valid_cols]
            .values.astype(float)
            .reshape(1, -1)
        )

        packed_scaled = scaler.transform(packed_feats)[0]
        X_test_packed_scaled[i] = packed_scaled
        paired_mask[i] = True

    else:
        # ---- FALLBACK: KEEP ORIGINAL RANSOMWARE FEATURES ----
        X_test_packed_scaled[i] = X_test_scaled[i]

print("Packed ransomware successfully replaced:", paired_mask.sum())
print("Ransomware fallback (unpacked kept):", np.sum((y_test == 1) & ~paired_mask))


# ------------------------------------------------------------
# 5. Compute ASR + Packed Accuracy
# ------------------------------------------------------------
def compute_asr(model, y_test, y_pred_unpacked, X_test_packed_scaled):
    y_pred_packed = model.predict(X_test_packed_scaled)
    rw_mask = (y_test == 1)

    correct_before = (y_pred_unpacked == 1) & rw_mask
    wrong_after = (y_pred_packed != 1) & rw_mask

    if correct_before.sum() == 0:
        return 0.0, y_pred_packed

    asr = (correct_before & wrong_after).sum() / correct_before.sum()
    return float(asr), y_pred_packed

rf_asr, rf_y_pred_packed = compute_asr(rf,  y_test, rf_y_pred,  X_test_packed_scaled)
svm_asr, svm_y_pred_packed = compute_asr(svm, y_test, svm_y_pred, X_test_packed_scaled)
mlp_asr, mlp_y_pred_packed = compute_asr(mlp, y_test, mlp_y_pred, X_test_packed_scaled)
xgb_asr, xgb_y_pred_packed = compute_asr(xgb, y_test, xgb_y_pred, X_test_packed_scaled)

comparison = pd.DataFrame([
    {"model": "RandomForest", "accuracy_unpacked": rf_metrics["accuracy"],
     "accuracy_packed": accuracy_score(y_test, rf_y_pred_packed),  "ASR": rf_asr},
    {"model": "SVM_RBF",      "accuracy_unpacked": svm_metrics["accuracy"],
     "accuracy_packed": accuracy_score(y_test, svm_y_pred_packed), "ASR": svm_asr},
    {"model": "MLP",          "accuracy_unpacked": mlp_metrics["accuracy"],
     "accuracy_packed": accuracy_score(y_test, mlp_y_pred_packed), "ASR": mlp_asr},
    {"model": "XGBoost",      "accuracy_unpacked": xgb_metrics["accuracy"],
     "accuracy_packed": accuracy_score(y_test, xgb_y_pred_packed), "ASR": xgb_asr},
])

comparison


Ransomware in test set: 2872
Packing ransomware...
Packed or reused: 1347
Packed ransomware successfully replaced: 1347
Ransomware fallback (unpacked kept): 1525


,model,accuracy_unpacked,accuracy_packed,ASR
0,RandomForest,0.983806,0.983806,0.000000
1,SVM_RBF,0.977190,0.977886,0.000355
2,MLP,0.983980,0.984155,0.000709
3,XGBoost,0.988682,0.988682,0.000000
